# 00 · Kiểm tra môi trường — Đức Tâm — XLM-R-base (kiến trúc 2)

**Đồ án NLP · Đề 26 · PhoNER_COVID19** · Issue **T05** · hạn **18/10/2026**

Notebook này trả lời 1 câu hỏi trước khi fine-tune: *"Máy của mình có chạy được thí nghiệm của nhóm một cách **tái lập được** không?"*
Nó kiểm tra 6 thứ và ghi kết quả ra file `results/moi_truong/tam.json` — chính là số liệu cho **mục (4) của Báo cáo tiến độ 1** ("môi trường đã setup").

| # | Kiểm tra | Vì sao cần |
|---|---|---|
| 1 | GPU, CUDA, phiên bản thư viện | Ghi lại môi trường để người khác chạy lại ra cùng kết quả |
| 2 | Lấy code nhóm + dữ liệu | Mọi người dùng cùng 1 bản code và dữ liệu |
| 3 | Số câu, nhãn, độ dài từ = nhãn | Phát hiện lỗi dữ liệu trước khi train |
| 4 | "Dấu vân tay" SHA-256 của dữ liệu | Chứng minh 4 người dùng **đúng cùng một** dữ liệu |
| 5 | Seed có tác dụng | Không có thì mean ± std qua 3 seed là vô nghĩa |
| 6 | Tokenizer + chạy thử model trên GPU | Biết số câu bị cắt ở 256 subword và ước lượng thời gian 1 epoch |

**Cách chạy (VS Code):** *File → Open Folder* → chọn thư mục `NLP-phoner-covid19-26` · mở notebook này · *Select Kernel* (môi trường Python đã cài thư viện) · **Run All** (~10 phút nếu có GPU).
**Không cần token GitHub** — notebook tự tải dữ liệu công khai của VinAI vào `data/` và đọc cấu hình chung từ `configs/common.json` của repo.

**Dựa trên thực hành được giới nghiên cứu công nhận:** báo cáo phân phối điểm qua nhiều seed thay vì 1 con số (Reimers & Gurevych, EMNLP 2017 — chính trên bài toán gán nhãn chuỗi/NER); ghi rõ môi trường, phần cứng và ngân sách tính toán (Dodge et al., EMNLP 2019 "Show Your Work"); checklist tái lập của NeurIPS (Pineau et al., JMLR 2021).

> Bạn khác dùng lại notebook này: chỉ sửa 4 dòng ở ô **Cấu hình** (TEN, VAI_TRO, MODEL_KIEM_TRA, MUC_DU_LIEU) rồi lưu thành `00_setup_<ten>.ipynb`.

## Khởi tạo môi trường (VS Code)
Chạy 2 ô dưới trước: ô đầu tìm thư mục gốc repo và bật `autoreload`; ô sau (đang để dạng chú thích) dùng khi máy còn thiếu thư viện.

In [ ]:
# Ô khởi tạo - chạy ĐẦU TIÊN. Notebook nằm trong notebooks/, còn code/dữ liệu/kết quả tính từ THƯ MỤC GỐC repo,
# nên ô này tìm thư mục gốc rồi chuyển về đó (đúng dù VS Code mở notebook với thư mục làm việc nào).
%load_ext autoreload
%autoreload 2

import os, sys, json, glob, shutil, time
from pathlib import Path


def tim_goc_repo():
    """Đi từ thư mục hiện tại lên trên tới khi gặp thư mục có configs/common.json và src/."""
    hien_tai = Path.cwd().resolve()
    for p in [hien_tai, *hien_tai.parents]:
        if (p / "configs" / "common.json").is_file() and (p / "src").is_dir():
            return p
    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc repo (cần có configs/common.json và src/). "
        "Trong VS Code: File -> Open Folder -> chọn thư mục NLP-phoner-covid19-26, rồi mở lại notebook.")


GOC_REPO = tim_goc_repo()
os.chdir(GOC_REPO)                                    # mọi đường dẫn tương đối (configs/, data/, results/, src/) tính từ gốc repo
if str(GOC_REPO / "src") not in sys.path:
    sys.path.insert(0, str(GOC_REPO / "src"))         # để `import data`, `import train_ner`, ...
print("Thư mục gốc repo :", GOC_REPO)
print("Python (kernel)  :", sys.executable)

In [ ]:
# Thiếu thư viện? Bỏ dấu # ở dòng %pip rồi chạy 1 lần (%pip cài vào ĐÚNG môi trường của kernel đang chọn trong VS Code).
# - KHÔNG dùng "pip install -r requirements.txt" cho notebook này: dòng `seqeval` trong đó lỗi "metadata-generation-failed"
#   trên Python mới và làm hỏng cả lệnh cài. seqeval được src/nap_seqeval.py tự nạp (cần scikit-learn, có trong dòng dưới).
# - torch: nếu máy có GPU NVIDIA, cài bản CUDA theo https://pytorch.org/get-started/locally/ thay cho "torch" ở dòng dưới.
#%pip install -q torch transformers accelerate scikit-learn pandas matplotlib ipywidgets

## 0. Cấu hình

In [ ]:
TEN = "tam"                                   # phuc | duyen | tam | trinh
VAI_TRO = "Đức Tâm — XLM-R-base (kiến trúc 2)"
MODEL_KIEM_TRA = "xlm-roberta-base"                      # vinai/phobert-base | xlm-roberta-base | bert-base-multilingual-cased
MUC_DU_LIEU = "syllable"                           # word (PhoBERT, CRF) | syllable (XLM-R, mBERT)

MAX_LENGTH = 256                                # theo configs/common.json


# seqeval (chấm điểm NER): "pip install seqeval" hay lỗi "metadata-generation-failed" trên Python mới (3.12+).
# src/nap_seqeval.py dùng bản cài sẵn nếu có, không thì tự tải mã nguồn thuần Python từ PyPI (không cần git/pip).
from nap_seqeval import dam_bao_seqeval
print("seqeval OK:", dam_bao_seqeval())
print("Cấu hình xong cho:", TEN, "|", MODEL_KIEM_TRA, "|", MUC_DU_LIEU)

## 1. Phần cứng: có GPU không, GPU gì

In [ ]:
import os, sys, json, time, platform, datetime
import torch

print("Python  :", sys.version.split()[0])
print("PyTorch :", torch.__version__, "| CUDA build:", torch.version.cuda)
CO_GPU = torch.cuda.is_available()
if CO_GPU:
    GPU_TEN = torch.cuda.get_device_name(0)
    VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
    print(f"GPU     : {GPU_TEN} · {VRAM_GB} GB")
else:
    GPU_TEN, VRAM_GB = "Không có GPU", 0
    print("⚠️ Không thấy GPU CUDA. Cần driver NVIDIA + bản torch có CUDA (https://pytorch.org/get-started/locally/), rồi Restart kernel.\n"
          "   Không có GPU vẫn chạy được mục 2-5; mục 6 (chạy thử model) sẽ chậm và mục 7 sẽ báo CHƯA ĐẠT ở dòng \"Có GPU\".")

## 2. Dữ liệu PhoNER_COVID19
Dùng cấu hình chung `configs/common.json` của repo và tải dữ liệu công khai của VinAI vào `data/` (**không commit** dữ liệu — giấy phép). Checkpoint (khi fine-tune) để ở `checkpoints/`, đã nằm trong `.gitignore`.

In [ ]:
TREN_COLAB = False       # notebook này chạy trên máy cá nhân (VS Code), không dùng Google Drive
DRIVE_OK = False         # giữ khóa "google_drive" trong results/moi_truong/<ten>.json cho cùng định dạng với file của cả nhóm
assert os.path.isfile("configs/common.json"), "Thiếu configs/common.json - hãy mở đúng thư mục gốc repo"
print("Thư mục làm việc:", os.getcwd())

In [ ]:
# Dữ liệu công khai của VinAI (PhoNER_COVID19) - chỉ dùng cho nghiên cứu/học tập, KHÔNG commit (đã chặn trong .gitignore).
# Hàm này tải thẳng file zip từ GitHub (không cần cài git) và chỉ tải khi data/ chưa đủ 6 file.
from data import tai_ve_du_lieu
tai_ve_du_lieu()
print(sorted(os.listdir("data")))

In [ ]:
import importlib.metadata as md

THU_VIEN = {}
for goi in ["torch", "transformers", "accelerate", "seqeval", "sklearn-crfsuite", "numpy", "pandas"]:
    try:
        THU_VIEN[goi] = md.version(goi)
    except md.PackageNotFoundError:
        THU_VIEN[goi] = "chưa cài"
for k, v in THU_VIEN.items():
    print(f"{k:18s} {v}")

## 3. Kiểm tra dữ liệu
Đọc cả 2 bản (mức từ và âm tiết) × 3 tập, so với số câu mong đợi trong `configs/common.json`, đếm câu có số từ ≠ số nhãn, nhãn lạ, nhãn `I-` đứng sai chỗ, và số thực thể.

In [ ]:
import pandas as pd
from seqeval.metrics.sequence_labeling import get_entities      # đã nạp ở ô 0

CAU_HINH = json.load(open("configs/common.json", encoding="utf-8"))
NHAN_HOP_LE = set(CAU_HINH["nhan"])
SO_CAU_MONG_DOI = CAU_HINH["du_lieu"]["so_cau_mong_doi"]


def doc_file_json(duong_dan):
    """Đọc 1 file .json PhoNER_COVID19 -> (danh sách câu, danh sách nhãn)."""
    cac_cau, cac_nhan = [], []
    with open(duong_dan, encoding="utf-8") as f:
        for dong in f:
            if dong.strip():
                mau = json.loads(dong)
                cac_cau.append(mau["words"])
                cac_nhan.append(mau["tags"])
    return cac_cau, cac_nhan


def dem_I_sai_cho(nhan_cau):
    """Đếm nhãn I-X không đứng sau B-X hoặc I-X (vi phạm IOB2)."""
    dem, truoc = 0, "O"
    for n in nhan_cau:
        if n.startswith("I-") and truoc[2:] != n[2:]:
            dem += 1
        truoc = n
    return dem


DU_LIEU, BANG = {}, []
for muc in ["word", "syllable"]:
    for split in ["train", "dev", "test"]:
        cac_cau, cac_nhan = doc_file_json(f"data/{muc}/{split}_{muc}.json")
        DU_LIEU[(muc, split)] = (cac_cau, cac_nhan)
        BANG.append({
            "bản": muc, "tập": split, "số câu": len(cac_cau),
            "mong đợi": SO_CAU_MONG_DOI[split],
            "câu lệch từ/nhãn": sum(len(c) != len(n) for c, n in zip(cac_cau, cac_nhan)),
            "nhãn lạ": len({x for n in cac_nhan for x in n} - NHAN_HOP_LE),
            "I- sai chỗ": sum(dem_I_sai_cho(n) for n in cac_nhan),
            "số thực thể": sum(len(get_entities(n)) for n in cac_nhan),
            "token TB/câu": round(sum(map(len, cac_cau)) / len(cac_cau), 1),
        })
BANG = pd.DataFrame(BANG)
DU_LIEU_OK = bool((BANG["số câu"] == BANG["mong đợi"]).all() and (BANG["câu lệch từ/nhãn"] == 0).all()
                  and (BANG["nhãn lạ"] == 0).all())
print("Dữ liệu đúng:", "✅" if DU_LIEU_OK else "❌")
BANG

In [ ]:
# Ví dụ 1 câu có nhiều loại thực thể (dùng cho mục "ví dụ mẫu dữ liệu" của BC1)
cac_cau, cac_nhan = DU_LIEU[(MUC_DU_LIEU, "train")]
i = max(range(200), key=lambda k: len({t[2:] for t in cac_nhan[k] if t != "O"}))
for tu, nhan in zip(cac_cau[i], cac_nhan[i]):
    print(f"{tu:25s} {nhan}")

## 4. Dấu vân tay dữ liệu (SHA-256)
Nếu 4 người có cùng dấu vân tay thì chắc chắn đang dùng **cùng một** dữ liệu — một bằng chứng tái lập được mà giảng viên thường hỏi.

In [ ]:
import glob, hashlib

def sha256(duong_dan):
    h = hashlib.sha256()
    with open(duong_dan, "rb") as f:
        for khoi in iter(lambda: f.read(1 << 20), b""):
            h.update(khoi)
    return h.hexdigest()

VAN_TAY = {p.replace(os.sep, "/"): sha256(p)[:16] for p in sorted(glob.glob("data/*/*.json"))}
for p, h in VAN_TAY.items():
    print(f"{p:35s} {h}")

## 5. Seed có tác dụng không
Cùng seed phải ra cùng số ngẫu nhiên, khác seed phải ra số khác. Nếu sai, 3 seed sẽ cho kết quả y hệt nhau và std = 0 là vô nghĩa.

In [ ]:
from transformers import set_seed

set_seed(42); a = torch.rand(5)
set_seed(42); b = torch.rand(5)
set_seed(123); c = torch.rand(5)
SEED_OK = torch.equal(a, b) and not torch.equal(a, c)
print("Cùng seed giống nhau:", torch.equal(a, b), "| Khác seed khác nhau:", not torch.equal(a, c), "->", "✅" if SEED_OK else "❌")

## 6. Tokenizer và chạy thử model
**6a.** Mỗi từ bị tokenizer cắt thành mấy *subword*? Bao nhiêu câu dài hơn 256 subword (sẽ bị cắt, mất thực thể ở cuối câu)?
**6b.** Chạy thử 10 bước huấn luyện trên GPU để biết bộ nhớ dùng và **ước lượng thời gian 1 epoch**.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_KIEM_TRA)
print("Tokenizer:", type(tok).__name__, "| fast:", tok.is_fast)

def tach_subword(cac_tu):
    """Mã hóa từng từ rồi ghép (cách an toàn cho cả PhoBERT - tokenizer chậm)."""
    return [tok.encode(tu, add_special_tokens=False) for tu in cac_tu]

cac_cau, cac_nhan = DU_LIEU[(MUC_DU_LIEU, "train")]
for tu, manh in list(zip(cac_cau[0], tach_subword(cac_cau[0])))[:12]:
    print(f"{tu:22s} -> {tok.convert_ids_to_tokens(manh)}")

do_dai = [2 + sum(len(m) for m in tach_subword(c)) for c in cac_cau]      # +2 cho <s> và </s>
SO_CAU_VUOT = sum(d > MAX_LENGTH for d in do_dai)
THONG_KE_DO_DAI = {"trung_binh": round(sum(do_dai) / len(do_dai), 1), "lon_nhat": max(do_dai),
                   "so_cau_vuot_max_length": SO_CAU_VUOT, "ti_le_vuot": round(SO_CAU_VUOT / len(do_dai), 4)}
print("\nĐộ dài subword trên train:", THONG_KE_DO_DAI)

In [ ]:
from transformers import AutoModelForTokenClassification

label2id = {n: i for i, n in enumerate(CAU_HINH["nhan"])}
thiet_bi = torch.device("cuda" if CO_GPU else "cpu")
model = AutoModelForTokenClassification.from_pretrained(MODEL_KIEM_TRA, num_labels=len(label2id)).to(thiet_bi)
toi_uu = torch.optim.AdamW(model.parameters(), lr=CAU_HINH["sieu_tham_so"]["learning_rate"])
BATCH = CAU_HINH["sieu_tham_so"]["batch_size"]


def tao_batch(chi_so):
    """Ghép batch: nhãn thật ở subword đầu của mỗi từ, -100 ở subword còn lại (bị bỏ qua khi tính loss)."""
    ds_id, ds_nhan = [], []
    for k in chi_so:
        ids, nhan = [tok.cls_token_id], [-100]
        for manh, n in zip(tach_subword(cac_cau[k]), cac_nhan[k]):
            ids += manh
            nhan += [label2id[n]] + [-100] * (len(manh) - 1)
        ids, nhan = ids[:MAX_LENGTH - 1] + [tok.sep_token_id], nhan[:MAX_LENGTH - 1] + [-100]
        ds_id.append(ids); ds_nhan.append(nhan)
    dai = max(map(len, ds_id))
    pad = lambda x, v: x + [v] * (dai - len(x))
    input_ids = torch.tensor([pad(x, tok.pad_token_id) for x in ds_id], device=thiet_bi)
    return input_ids, (input_ids != tok.pad_token_id).long(), torch.tensor([pad(x, -100) for x in ds_nhan], device=thiet_bi)


SO_BUOC = 10 if CO_GPU else 2
scaler = torch.amp.GradScaler("cuda", enabled=CO_GPU)
model.train()
if CO_GPU:
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
bat_dau = time.time()
for buoc in range(SO_BUOC):
    ids, mask, nhan = tao_batch(range(buoc * BATCH, (buoc + 1) * BATCH))
    with torch.autocast(device_type=thiet_bi.type, dtype=torch.float16, enabled=CO_GPU):
        loss = model(input_ids=ids, attention_mask=mask, labels=nhan).loss
    scaler.scale(loss).backward(); scaler.step(toi_uu); scaler.update(); toi_uu.zero_grad()
    print(f"bước {buoc + 1:2d}  loss = {loss.item():.4f}")
if CO_GPU:
    torch.cuda.synchronize()
GIAY_MOI_BUOC = (time.time() - bat_dau) / SO_BUOC
SO_BUOC_MOI_EPOCH = -(-len(cac_cau) // BATCH)
UOC_LUONG = {
    "giay_moi_buoc": round(GIAY_MOI_BUOC, 3),
    "phut_moi_epoch_train": round(GIAY_MOI_BUOC * SO_BUOC_MOI_EPOCH / 60, 1),
    "phut_toi_da_1_seed": round(GIAY_MOI_BUOC * SO_BUOC_MOI_EPOCH * CAU_HINH["sieu_tham_so"]["so_epoch_toi_da"] / 60, 1),
    "vram_dinh_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 2) if CO_GPU else None,
}
MODEL_OK = bool(torch.isfinite(loss).item())
print("\nƯớc lượng:", UOC_LUONG, "(chưa tính thời gian chấm dev mỗi epoch)")

## 7. Ghi báo cáo môi trường + tự kiểm tra "Xong khi"

In [ ]:
BAO_CAO = {
    "nguoi": TEN, "vai_tro": VAI_TRO, "ngay_chay": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "nen_tang": "Colab" if TREN_COLAB else platform.platform(),
    "gpu": GPU_TEN, "vram_GB": VRAM_GB, "python": sys.version.split()[0], "thu_vien": THU_VIEN,
    "model_kiem_tra": MODEL_KIEM_TRA, "muc_du_lieu": MUC_DU_LIEU,
    "du_lieu": BANG.to_dict(orient="records"), "van_tay_du_lieu_sha256_16": VAN_TAY,
    "seed_hoat_dong": SEED_OK, "google_drive": DRIVE_OK, "do_dai_subword_train": THONG_KE_DO_DAI, "uoc_luong_thoi_gian": UOC_LUONG,
}
os.makedirs("results/moi_truong", exist_ok=True)
with open(f"results/moi_truong/{TEN}.json", "w", encoding="utf-8") as f:
    json.dump(BAO_CAO, f, ensure_ascii=False, indent=2)
print(f"Đã ghi results/moi_truong/{TEN}.json\n")

KIEM_TRA = {
    "Có GPU (T4/P100/...)": CO_GPU,
    "Đọc đủ train/dev/test = 5.027 / 2.000 / 3.000 câu, cả 2 bản, không lệch, không nhãn lạ": DU_LIEU_OK,
    "Seed có tác dụng": SEED_OK,
    "Model chạy được forward + backward, loss hữu hạn": MODEL_OK,
}
for y, ok in KIEM_TRA.items():
    print("✅" if ok else "❌", y)
print("\n=> KẾT LUẬN:", "ĐẠT — mở Pull Request" if all(KIEM_TRA.values()) else "CHƯA ĐẠT — xem dòng ❌")


## 8. Nộp bằng Pull Request

Notebook đã nằm đúng chỗ (`notebooks/00_setup_tam.ipynb`) và file kết quả đã tự ghi vào `results/moi_truong/tam.json` — không cần tải hay chép thêm gì.

1. Kiểm tra đang ở nhánh `tam/T05-setup` (tạo bằng GitHub Desktop hoặc `git switch -c tam/T05-setup`).
2. Trước khi commit: *Restart* kernel → **Run All** để chắc chạy từ đầu không lỗi (quy ước ở `notebooks/README.md`).
3. GitHub Desktop: Commit `T05: notebook kiểm tra môi trường` → **Publish branch** → **Create Pull Request** → ghi `Closes #<số Issue T05>`, dán kết quả ô 7.